In [2]:
import os
import joblib
import lightgbm as lgb

model_dir = r"d://Ahmed//Credit_Loan_Risk//models"
model_path_pkl = os.path.join(model_dir, "lgbm_model.pkl")

lgbm = joblib.load(model_path_pkl)


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("D:/Ahmed/Credit_Loan_Risk")

X_train = pd.read_csv(PROJECT_ROOT / "Dataset" / "X_train.csv")
X_test  = pd.read_csv(PROJECT_ROOT / "Dataset" / "X_test.csv")
y_train = pd.read_csv(PROJECT_ROOT / "Dataset" / "y_train.csv").squeeze()
y_test  = pd.read_csv(PROJECT_ROOT / "Dataset" / "y_test.csv").squeeze()
scaler  = joblib.load(PROJECT_ROOT / "Dataset" / "robust_scaler.pkl")

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train default rate: {y_train.mean():.4f}")

X_train: (246005, 64), X_test: (61502, 64)
y_train default rate: 0.0807


In [13]:
# ------------------------------------------------------------------
# Cell 1 — SHAP setup and values
# ------------------------------------------------------------------
import shap

explainer = shap.TreeExplainer(lgbm)

sample_idx = X_test.sample(5000, random_state=42).index
X_test_sample = X_test.loc[sample_idx]

shap_values = explainer.shap_values(X_test_sample)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Base rate (expected value): {explainer.expected_value:.4f}")

SHAP values shape: (5000, 64)
Base rate (expected value): -2.6439


d:\Ahmed\Credit_Loan_Risk\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [14]:
# ------------------------------------------------------------------
# Cell 2 — Global feature importance (text output)
# ------------------------------------------------------------------
# Mean absolute SHAP value per feature = average impact on prediction
shap_importance = pd.DataFrame({
    "feature"         : X_test_sample.columns,
    "mean_abs_shap"   : np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

print("=== Global Feature Importance (SHAP) ===")
print("Mean absolute SHAP value = average impact on prediction across all customers")
print("Higher = more influential feature\n")
print(shap_importance.to_string(index=False))

=== Global Feature Importance (SHAP) ===
Mean absolute SHAP value = average impact on prediction across all customers
Higher = more influential feature

                                feature  mean_abs_shap
                           EXT_SOURCE_2       0.317714
                           EXT_SOURCE_3       0.301872
                            CREDIT_TERM       0.171649
                   EXT_SOURCE_1_IMPUTED       0.137091
                  CREDIT_TO_GOODS_RATIO       0.123897
                    NAME_EDUCATION_TYPE       0.106226
                            CODE_GENDER       0.096360
                      ORGANIZATION_TYPE       0.092054
                            AMT_ANNUITY       0.081834
                          DAYS_EMPLOYED       0.077353
                             DAYS_BIRTH       0.072294
                 BUREAU_AMT_CREDIT_MEAN       0.067455
             NAME_FAMILY_STATUS_Married       0.063506
                    BUREAU_ACTIVE_RATIO       0.062955
                      

In [15]:
# ------------------------------------------------------------------
# Cell 3 — Direction of impact per feature
# ------------------------------------------------------------------
print("=== Feature Impact Direction ===")
print("For each top feature: does a HIGH value push prediction UP or DOWN?\n")

top_features = shap_importance["feature"].head(15).tolist()

for feat in top_features:
    feat_idx = list(X_test_sample.columns).index(feat)
    feat_values = X_test_sample[feat].values
    feat_shap   = shap_values[:, feat_idx]
    
    # Correlation between feature value and its SHAP value
    # Positive = higher feature value pushes prediction UP (more default risk)
    # Negative = higher feature value pushes prediction DOWN (less default risk)
    corr = np.corrcoef(feat_values, feat_shap)[0, 1]
    direction = "↑ higher value = MORE default risk" if corr > 0 else "↓ higher value = LESS default risk"
    print(f"  {feat:<35} corr={corr:+.3f}  {direction}")

=== Feature Impact Direction ===
For each top feature: does a HIGH value push prediction UP or DOWN?

  EXT_SOURCE_2                        corr=-0.975  ↓ higher value = LESS default risk
  EXT_SOURCE_3                        corr=-0.976  ↓ higher value = LESS default risk
  CREDIT_TERM                         corr=-0.064  ↓ higher value = LESS default risk
  EXT_SOURCE_1_IMPUTED                corr=-0.934  ↓ higher value = LESS default risk
  CREDIT_TO_GOODS_RATIO               corr=+0.916  ↑ higher value = MORE default risk
  NAME_EDUCATION_TYPE                 corr=-0.972  ↓ higher value = LESS default risk
  CODE_GENDER                         corr=+0.952  ↑ higher value = MORE default risk
  ORGANIZATION_TYPE                   corr=+0.929  ↑ higher value = MORE default risk
  AMT_ANNUITY                         corr=+0.705  ↑ higher value = MORE default risk
  DAYS_EMPLOYED                       corr=-0.827  ↓ higher value = LESS default risk
  DAYS_BIRTH                          

In [19]:
# ------------------------------------------------------------------
# Cell 4 revised — Pick a cleaner high risk customer
# ------------------------------------------------------------------
# Find a high risk customer where EXT_SOURCE values are present
# (not missing/imputed) for a cleaner explanation

y_proba_sample = lgbm.predict_proba(X_test_sample)[:, 1]
sample_df = X_test_sample.copy()
sample_df["proba"] = y_proba_sample

# High risk + has real EXT_SOURCE values (not extreme scaled values)
ext2_idx  = list(X_test_sample.columns).index("EXT_SOURCE_2")
ext3_idx  = list(X_test_sample.columns).index("EXT_SOURCE_3")

clean_mask = (
    (sample_df["proba"] > 0.40) &                        # genuinely high risk
    (sample_df["EXT_SOURCE_2"] > -2.0) &                 # not extreme outlier
    (sample_df["EXT_SOURCE_3"] > -2.0)
)

clean_candidates = sample_df[clean_mask].sort_values("proba", ascending=False)
print(f"Clean high-risk candidates: {len(clean_candidates)}")
print(clean_candidates[["proba", "EXT_SOURCE_2", "EXT_SOURCE_3"]].head(10))

sample_pos = X_test_sample.index.get_loc(clean_candidates.index[0])

customer_shap = shap_values[sample_pos]
customer_data = X_test_sample.iloc[sample_pos]
customer_prob = clean_candidates["proba"].iloc[0]

print(f"\n=== High Risk Customer Explanation ===")
print(f"Predicted default probability : {customer_prob:.4f} ({customer_prob*100:.1f}%)")
print(f"\nTop features driving THIS customer's prediction:")
print(f"{'Feature':<35} {'Scaled Value':>12} {'SHAP':>10} {'Direction':>25}")
print("-" * 90)

feat_df = pd.DataFrame({
    "feature" : X_test_sample.columns,
    "value"   : customer_data.values,
    "shap"    : customer_shap
}).reindex(pd.Series(np.abs(customer_shap)).sort_values(ascending=False).index)

for _, row in feat_df.head(15).iterrows():
    direction = "↑ increases default risk" if row["shap"] > 0 else "↓ decreases default risk"
    print(f"  {row['feature']:<33} {row['value']:>12.3f} {row['shap']:>+10.4f}  {direction}")

Clean high-risk candidates: 68
          proba  EXT_SOURCE_2  EXT_SOURCE_3
24587  0.613819     -1.691888      0.000000
47958  0.613797     -1.141934     -1.413856
32435  0.608924     -1.840057      0.000000
43117  0.580067     -1.948984     -1.744378
41694  0.564247     -1.551055      0.000000
34459  0.560005     -1.709701     -0.575231
22914  0.558980     -1.226872      0.000000
52547  0.556680     -1.078435     -1.704283
36528  0.545238     -1.087460      0.000000
5011   0.541643     -1.706107     -0.289620

=== High Risk Customer Explanation ===
Predicted default probability : 0.6138 (61.4%)

Top features driving THIS customer's prediction:
Feature                             Scaled Value       SHAP                 Direction
------------------------------------------------------------------------------------------
  EXT_SOURCE_2                            -1.692    +0.9658  ↑ increases default risk
  CREDIT_TO_GOODS_RATIO                    1.000    +0.3348  ↑ increases default risk

In [17]:
# ------------------------------------------------------------------
# Cell 5 — Probability distribution (text)
# ------------------------------------------------------------------
y_proba_full = lgbm.predict_proba(X_test)[:, 1]

print("=== Probability Distribution Percentiles ===")
percentiles = [5, 10, 25, 50, 75, 90, 95, 99]
for p in percentiles:
    val = pd.Series(y_proba_full).quantile(p/100)
    print(f"  {p:>3}th percentile: {val:.4f} ({val*100:.2f}%)")

print(f"\n=== Population above key thresholds ===")
thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
for t in thresholds:
    count = (y_proba_full > t).sum()
    pct   = (y_proba_full > t).mean() * 100
    print(f"  Above {t:.2f}: {count:>6} customers ({pct:.1f}%)")

print(f"\n=== Default rate within predicted risk bands ===")
y_test_arr = y_test.values
bands = [(0.00, 0.10), (0.10, 0.20), (0.20, 0.30), (0.30, 1.00)]
for low, high in bands:
    mask        = (y_proba_full >= low) & (y_proba_full < high)
    count       = mask.sum()
    actual_rate = y_test_arr[mask].mean() if count > 0 else 0
    print(f"  Prob [{low:.2f}-{high:.2f}): {count:>6} customers, actual default rate: {actual_rate:.3f} ({actual_rate*100:.1f}%)")

=== Probability Distribution Percentiles ===
    5th percentile: 0.0141 (1.41%)
   10th percentile: 0.0195 (1.95%)
   25th percentile: 0.0334 (3.34%)
   50th percentile: 0.0629 (6.29%)
   75th percentile: 0.1231 (12.31%)
   90th percentile: 0.2187 (21.87%)
   95th percentile: 0.2949 (29.49%)
   99th percentile: 0.4649 (46.49%)

=== Population above key thresholds ===
  Above 0.05:  36556 customers (59.4%)
  Above 0.10:  19727 customers (32.1%)
  Above 0.15:  11634 customers (18.9%)
  Above 0.20:   7275 customers (11.8%)
  Above 0.25:   4685 customers (7.6%)
  Above 0.30:   2931 customers (4.8%)
  Above 0.40:   1158 customers (1.9%)
  Above 0.50:    413 customers (0.7%)

=== Default rate within predicted risk bands ===
  Prob [0.00-0.10):  41775 customers, actual default rate: 0.037 (3.7%)
  Prob [0.10-0.20):  12452 customers, actual default rate: 0.117 (11.7%)
  Prob [0.20-0.30):   4344 customers, actual default rate: 0.213 (21.3%)
  Prob [0.30-1.00):   2931 customers, actual default r

**______________________________________________________________________________________________________**

**RISK SEGMENTATION**

In [34]:

# ------------------------------------------------------------------
# Assign risk categories
# ------------------------------------------------------------------
y_proba_full = lgbm.predict_proba(X_test)[:, 1]

def assign_risk(prob):
    if prob < 0.10:   return "Low Risk"
    elif prob < 0.3: return "Medium Risk"
    else:             return "High Risk"

risk_df = pd.DataFrame({
    "probability"    : y_proba_full,
    "risk_category"  : [assign_risk(p) for p in y_proba_full],
    "actual_default" : y_test.values
}, index=X_test.index)

print("=== Risk Category Distribution ===")
print(risk_df["risk_category"].value_counts().to_string())
print(f"\nTotal customers: {len(risk_df)}")

=== Risk Category Distribution ===
risk_category
Low Risk       41775
Medium Risk    16796
High Risk       2931

Total customers: 61502


In [35]:
# ------------------------------------------------------------------
# Cell 2 — Segment analysis
# ------------------------------------------------------------------
print("=== Segment Analysis ===\n")

for segment in ["Low Risk", "Medium Risk", "High Risk"]:
    seg = risk_df[risk_df["risk_category"] == segment]
    count        = len(seg)
    pct_pop      = count / len(risk_df) * 100
    actual_dr    = seg["actual_default"].mean()
    avg_prob     = seg["probability"].mean()
    caught       = seg["actual_default"].sum()

    print(f"--- {segment} ---")
    print(f"  Population        : {count:>6} customers ({pct_pop:.1f}% of total)")
    print(f"  Avg predicted prob: {avg_prob:.3f} ({avg_prob*100:.1f}%)")
    print(f"  Actual default rate: {actual_dr:.3f} ({actual_dr*100:.1f}%)")
    print(f"  Actual defaulters : {int(caught):>6} customers")
    print()

=== Segment Analysis ===

--- Low Risk ---
  Population        :  41775 customers (67.9% of total)
  Avg predicted prob: 0.046 (4.6%)
  Actual default rate: 0.037 (3.7%)
  Actual defaulters :   1563 customers

--- Medium Risk ---
  Population        :  16796 customers (27.3% of total)
  Avg predicted prob: 0.167 (16.7%)
  Actual default rate: 0.142 (14.2%)
  Actual defaulters :   2378 customers

--- High Risk ---
  Population        :   2931 customers (4.8% of total)
  Avg predicted prob: 0.401 (40.1%)
  Actual default rate: 0.349 (34.9%)
  Actual defaulters :   1024 customers



In [36]:
# ------------------------------------------------------------------
# Cell 3 — Business impact analysis
# ------------------------------------------------------------------
print("=== Business Impact Analysis ===\n")

total_defaulters = risk_df["actual_default"].sum()
total_customers  = len(risk_df)

for segment in ["Low Risk", "Medium Risk", "High Risk"]:
    seg     = risk_df[risk_df["risk_category"] == segment]
    caught  = seg["actual_default"].sum()
    missed  = seg[seg["actual_default"] == 1].shape[0]

    print(f"--- {segment} ---")
    if segment == "Low Risk":
        print(f"  Action            : Auto Approve")
        print(f"  Defaulters approved by mistake : {int(caught)} ({caught/total_defaulters*100:.1f}% of all defaulters)")
        print(f"  Good customers approved        : {int((seg['actual_default']==0).sum())}")
    elif segment == "Medium Risk":
        print(f"  Action            : Manual Review")
        print(f"  Defaulters sent to review      : {int(caught)} ({caught/total_defaulters*100:.1f}% of all defaulters)")
        print(f"  Good customers sent to review  : {int((seg['actual_default']==0).sum())}")
    else:
        print(f"  Action            : Reject / Extra Verification")
        print(f"  Defaulters correctly blocked   : {int(caught)} ({caught/total_defaulters*100:.1f}% of all defaulters)")
        print(f"  Good customers maybe incorrectly blocked or extra verification needed: {int((seg['actual_default']==0).sum())}")
    print()

print(f"Total defaulters in test set: {int(total_defaulters)}")

=== Business Impact Analysis ===

--- Low Risk ---
  Action            : Auto Approve
  Defaulters approved by mistake : 1563 (31.5% of all defaulters)
  Good customers approved        : 40212

--- Medium Risk ---
  Action            : Manual Review
  Defaulters sent to review      : 2378 (47.9% of all defaulters)
  Good customers sent to review  : 14418

--- High Risk ---
  Action            : Reject / Extra Verification
  Defaulters correctly blocked   : 1024 (20.6% of all defaulters)
  Good customers maybe incorrectly blocked or extra verification needed: 1907

Total defaulters in test set: 4965


In [39]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║                 CREDIT RISK LENDING STRATEGY                         ║
╠══════════════════════════════════════════════════════════════════════╣
║  SEGMENT    │ THRESHOLD    │ ACTION          │ RATIONALE             ║
╠══════════════════════════════════════════════════════════════════════╣
║  Low Risk   │ prob < 0.10  │ Auto Approve    │ 3.7% default rate     ║
║             │              │                 │ below 8% base rate    ║
╠══════════════════════════════════════════════════════════════════════╣
║  Medium     │ 0.10 - 0.30  │ Manual Review   │ 14.2% default rate    ║
║  Risk       │              │                 │ 1.8x base rate        ║
╠══════════════════════════════════════════════════════════════════════╣
║  High Risk  │ prob > 0.30  │ Reject / Extra  │ 34.9% default rate    ║
║             │              │ Verification    │ 4.3x base rate        ║
╚══════════════════════════════════════════════════════════════════════╝

Portfolio Summary (Test Set — 61,502 customers):
  Auto-approved  : 41,775 (67.9%) │ Expected defaults: 1,563 (3.7%)
  Manual review  : 16,796 (27.3%) │ Expected defaults: 2,378 (14.2%)
  Rejected/Verify:  2,931  (4.8%) │ Expected defaults: 1,024 (34.9%)
  
  Defaulters caught via review/rejection: 3,402 / 4,965 (68.5%)
  Good customers auto-approved          : 40,212 / 56,537 (71.1%)
""")


╔══════════════════════════════════════════════════════════════════════╗
║                 CREDIT RISK LENDING STRATEGY                         ║
╠══════════════════════════════════════════════════════════════════════╣
║  SEGMENT    │ THRESHOLD    │ ACTION          │ RATIONALE             ║
╠══════════════════════════════════════════════════════════════════════╣
║  Low Risk   │ prob < 0.10  │ Auto Approve    │ 3.7% default rate     ║
║             │              │                 │ below 8% base rate    ║
╠══════════════════════════════════════════════════════════════════════╣
║  Medium     │ 0.10 - 0.30  │ Manual Review   │ 14.2% default rate    ║
║  Risk       │              │                 │ 1.8x base rate        ║
╠══════════════════════════════════════════════════════════════════════╣
║  High Risk  │ prob > 0.30  │ Reject / Extra  │ 34.9% default rate    ║
║             │              │ Verification    │ 4.3x base rate        ║
╚═════════════════════════════════════════════════

**______________________________________________________________________________________________________**

**BUILDING THE PREPROCESSING FUNCTION**

In [40]:

PROJECT_ROOT = Path("D:/Ahmed/Credit_Loan_Risk")
for f in sorted((PROJECT_ROOT / "Dataset").iterdir()):
    print(f.name)

.DS_Store
applications_dataset.csv
bureau.csv
Credit Default Risk.pdf
dataset_description.xlsx
HomeCredit_columns_description.csv
non_NA_df.csv
previous_application.csv
robust_scaler.pkl
train_preview.csv
X_test.csv
X_train.csv
y_test.csv
y_train.csv


In [41]:
# ------------------------------------------------------------------
# Cell — Save all inference artifacts
# ------------------------------------------------------------------
import joblib
import json

# Save exact feature names and order the model expects
feature_names = X_train.columns.tolist()
with open(PROJECT_ROOT / "Dataset" / "feature_names.json", "w") as f:
    json.dump(feature_names, f)
print(f"✅ Saved: feature_names.json ({len(feature_names)} features)")

# 4. Save SHAP explainer
joblib.dump(explainer, PROJECT_ROOT / "Dataset" / "shap_explainer.pkl")
print("✅ Saved: shap_explainer.pkl")

# 5. Save risk segmentation thresholds
risk_thresholds = {
    "low_risk_max"    : 0.10,
    "medium_risk_max" : 0.30,
    "actions": {
        "Low Risk"    : "Auto Approve",
        "Medium Risk" : "Manual Review",
        "High Risk"   : "Reject / Extra Verification"
    }
}
with open(PROJECT_ROOT / "Dataset" / "risk_thresholds.json", "w") as f:
    json.dump(risk_thresholds, f)
print("✅ Saved: risk_thresholds.json")

print("\nAll artifacts saved.")

✅ Saved: feature_names.json (64 features)
✅ Saved: shap_explainer.pkl
✅ Saved: risk_thresholds.json

All artifacts saved.
